# Project 2 â€” Manufacturing Risk Experimentation Notebook

This notebook performs the experiment for the manufacturing project. It compares several classifier families on a machine-relative feature matrix and then stores the winning model as the one used by the production pipeline.

## Data cleaning and NaN policy

The manufacturing dataset is sensor telemetry, so the key handling rule is to interpolate short gaps in the temperature and vibration series because sensor readings should remain temporally continuous. A `NaN` can remain a `NaN` only when the model cannot build a valid machine-relative signal from the available history; otherwise, we repair it. This is why the z-score and rolling-window features are built after the data have been reindexed and interpolated, rather than by dropping the row outright.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RAW_PATH = Path("project2_manufacturing_sensors.csv")
NEW_MACHINES = {"MCH-300", "MCH-301"}
LOOKAHEAD_HOURS = 24
HOLDOUT_DAYS = 20
print("Setup complete.")


In [ ]:
df = pd.read_csv(RAW_PATH, parse_dates=["timestamp"])
print(f"Raw rows: {len(df)}")

before = len(df)
df = df.drop_duplicates(subset=["timestamp", "machine_id"], keep="first")
print(f"Duplicate rows dropped: {before - len(df)}")

df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)

cleaned = []
for mid, g in df.groupby("machine_id"):
    g = g.set_index("timestamp").sort_index()
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="h")
    g = g.reindex(full_idx)
    g["machine_id"] = mid
    g["line"] = g["line"].ffill().bfill()
    g["failure_event"] = g["failure_event"].fillna(0)
    g["run_hours_since_maintenance"] = g["run_hours_since_maintenance"].interpolate().bfill().ffill()
    g["temperature_c"] = g["temperature_c"].interpolate(limit=6).bfill().ffill()
    g["vibration_mm_s"] = g["vibration_mm_s"].interpolate(limit=6).bfill().ffill()
    cleaned.append(g)

clean = pd.concat(cleaned).rename_axis("timestamp").reset_index()
clean = clean.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
clean.info()


In [ ]:
baseline = clean.groupby("machine_id").agg(
    temp_base_mean=("temperature_c", "mean"), temp_base_std=("temperature_c", "std"),
    vib_base_mean=("vibration_mm_s", "mean"), vib_base_std=("vibration_mm_s", "std"),
).reset_index()
clean = clean.merge(baseline, on="machine_id", how="left")

for w in (6, 24):
    clean[f"temp_roll_mean_{w}"] = clean.groupby("machine_id")["temperature_c"].transform(
        lambda s: s.rolling(w, min_periods=1).mean())
    clean[f"vib_roll_mean_{w}"] = clean.groupby("machine_id")["vibration_mm_s"].transform(
        lambda s: s.rolling(w, min_periods=1).mean())

clean["temp_zscore"] = (clean["temp_roll_mean_6"] - clean["temp_base_mean"]) / clean["temp_base_std"]
clean["vib_zscore"] = (clean["vib_roll_mean_6"] - clean["vib_base_mean"]) / clean["vib_base_std"]
clean["temp_trend_6h"] = clean.groupby("machine_id")["temperature_c"].transform(lambda s: s.diff(6))
clean["vib_trend_6h"] = clean.groupby("machine_id")["vibration_mm_s"].transform(lambda s: s.diff(6))

FEATURES = [
    "temp_zscore", "vib_zscore", "temp_trend_6h", "vib_trend_6h",
    "temp_roll_mean_6", "vib_roll_mean_6", "temp_roll_mean_24", "vib_roll_mean_24",
    "run_hours_since_maintenance",
]

# Relabel as 24-hour failure-in-window label.
def label_lookahead(g):
    fail_times = g.loc[g.failure_event == 1, "timestamp"]
    y = pd.Series(0, index=g.index)
    for ft in fail_times:
        window = (g["timestamp"] < ft) & (g["timestamp"] >= ft - pd.Timedelta(hours=LOOKAHEAD_HOURS))
        y[window] = 1
    return y

clean["label_at_risk_24h"] = clean.groupby("machine_id", group_keys=False).apply(lambda g: label_lookahead(g))

model_df = clean.dropna(subset=FEATURES).copy()
established = model_df[~model_df.machine_id.isin(NEW_MACHINES)]
cutoff = established["timestamp"].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train = established[established["timestamp"] <= cutoff]
test = established[established["timestamp"] > cutoff]
print(train.shape, test.shape)


## Model comparison experiment

This experiment compares Random Forest, Extra Trees, Gradient Boosting, and Logistic Regression on the same time split and PR-AUC, because the positive class is genuinely rare here and PR-AUC is the right metric for the selected problem.


In [ ]:
pos_rate = train["label_at_risk_24h"].mean()

candidates = {
    "RandomForestClassifier": RandomForestClassifier(
        n_estimators=120, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=42,
    ),
    "ExtraTreesClassifier": ExtraTreesClassifier(
        n_estimators=120, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=42,
    ),
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)),
    ]),
}

results = []
for name, estimator in candidates.items():
    estimator.fit(train[FEATURES], train["label_at_risk_24h"])
    proba = estimator.predict_proba(test[FEATURES])[:, 1]
    score = average_precision_score(test["label_at_risk_24h"], proba)
    results.append({"model": name, "pr_auc": float(score)})
    print(f"{name}: PR-AUC={score:.4f}")

experiment_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
print("Notebook experiment ranking:")
print(experiment_df.to_string(index=False))

best_model_name = experiment_df.iloc[0]['model']
print(f"Selected model for pipeline: {best_model_name}")


In [ ]:
experiment_df.to_json("./output/p2_experiment_ranking.json", orient="records", indent=2)
print("Wrote p2_experiment_ranking.json")
